In [1]:
# """
# #**lstm_myopia_children.py**
# ────────────────────────────────────────────────────────────────────────────────
# Prédiction de la progression myopique chez les enfants (≤ 16 ans)
# Cibles  : SER_right et SER_left à 13 mois après la dernière visite
# Modèle  : LSTM bidirectionnel avec double sortie
# Données : 1 ligne par année de visite par patient (format long)
# ────────────────────────────────────────────────────────────────────────────────
# Utilisation
#     Entraînement  : python lstm_myopia_children.py --mode train
#     Prédiction    : python lstm_myopia_children.py --mode predict
# ════════════════════════════════════════════════════════════════════════════════
# """

In [2]:
from math import sqrt

import pandas as pd
import numpy as np
from pathlib import Path
import pyodbc 
import sqlalchemy
from sqlalchemy.engine import url
from sqlalchemy.engine.url import URL
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
#fit LSTM Model
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import keras
import tensorflow as tf
from tensorflow.keras.layers import LSTM
from keras.models import Sequential
from keras.layers import Dense



**Data Overview & Preprocessing**

In [3]:
p = Path.home()
p

In [27]:
df = pd.read_sql("SELECT * FROM forecasting_ChildrenAmetropia", engine)

In [28]:
df.head()

In [29]:
df.info()

In [30]:
df.shape

In [31]:
df.describe()

In [32]:
from summarytools import dfSummary
dfSummary(df,is_collapsible = True)

In [33]:
df_hyperopia = df[df['ametropia']=='hyperopia']
df_myopia = df[df['ametropia']=='myopia']
df_highmyopia = df[df['ametropia']=='high myopia']
df_premyopia = df[df['ametropia']=='pre-myopia']

In [34]:
data_hyperopia = df_hyperopia.groupby('age')[['age', 'SER_right', 'SER_left']].mean()
fig = px.line(data_hyperopia, x='age', y=['SER_right', 'SER_left'], markers=True)
fig.show()

In [36]:
data_myopia = df_myopia.groupby('age')[['age', 'SER_right', 'SER_left']].mean()
fig = px.line(data_myopia, x='age', y=['SER_right', 'SER_left'], markers=True)
fig.show()

In [38]:
data_highmyopia = df_highmyopia.groupby('age')[['age', 'SER_right', 'SER_left']].mean()
fig = px.line(data_highmyopia, x='age', y=['SER_right', 'SER_left'], markers=True)
fig.show()

In [40]:
fig = px.box(df, x="ametropia", y="SER_left")
fig.show()

In [41]:
fig = px.box(df, x="ametropia", y="SER_right")
fig.show()

In [42]:
data = df.groupby('ametropia')[['age', 'SER_right', 'SER_left']].mean()#.query("SER_right<0")
data.head()

In [46]:
df.columns

In [ ]:
# Conversion des colonnes "object" en valeurs numériques
label_encoder = LabelEncoder()
df['ametropia_encoded'] = label_encoder.fit_transform(df['ametropia'])
df['gender_encoded'] = label_encoder.fit_transform(df['Gender'])
df['Country_encoded'] = label_encoder.fit_transform(df['Country'])

In [51]:
df.head()

In [53]:
df.info()

In [54]:
df_cat= df.select_dtypes(include='object')
df_num = df.select_dtypes(include=['float', 'int'])

In [78]:
#data correlation
corr_matrix = df_num.corr(method='spearman', min_periods=1) #la méthose "Spearman" me paraît la plus pertinente 
                                                                    #pour obtenir des résultats optimaux. 
# sns.heatmap(corr_matrix, annot=False)
cmap = sns.diverging_palette(220, 10, as_cmap=True)
matrix = sns.heatmap(corr_matrix, cmap=cmap, cbar_kws={"shrink": .5}, linewidths=.5)
plt.show()

In [56]:
#visualise outliers
fig_scatter = px.scatter(df, x='age', y=['SER_right', 'SER_left'])
fig_scatter.show()

In [75]:
df1 = df[['age', 'SER_left', 'SER_right']]
df1.head()

In [76]:
df1.shape

In [77]:
values = df1.values
# specify columns to plot
groups = [1, 2]
i = 1
#plot each column
plt.figure()
for group in groups:
    plt.subplot(len(groups), 1, i)
    plt.plot(values[:, group])
    plt.title(df1.columns[group], y=0.5, loc='right')
    i += 1
plt.show()

In [ ]:
#Focus on myopia
df_myopia = df[df["ametropia"].isin(["myopia", "high myopia", "low myopia", "pre-myopia"])].reset_index(drop=True)

data preparation : coming

In [ ]:
df_myopia.isna().sum()

In [ ]:
#fillna for numeric column  

df_myopia['LeftAddition'] = df_myopia['LeftAddition'].fillna(0)
df_myopia['RightAddition'] = df_myopia['RightAddition'].fillna(0)
df_myopia['LeftCylinder'] = df_myopia['LeftCylinder'].fillna(0)
df_myopia['RightCylinder'] = df_myopia['RightCylinder'].fillna(0)
df_myopia['astigmatism'] = df_myopia['astigmatism'].fillna("missing")
df_myopia['LeftAxis'] = df_myopia['LeftAxis'].fillna(0)
df_myopia['RightAxis'] = df_myopia['RightAxis'].fillna(0)

df_mean_age = df_myopia['age'].mean()
df['age'] = df_myopia['age'].fillna(df_mean_age)

df_myopia['Country'] = df_myopia['Country'].fillna("missing")

In [ ]:
df_myopia.isna().sum()

Model : LSTM